In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
# from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score


In [ ]:
df = pd.read_csv('cardio_train.csv', sep=';')
df.sample(5)

In [ ]:
print(f"{df.shape[0]} rows and {df.shape[1]} columns in the dataset.")
print(f"Columns in the dataset: {df.columns.tolist()}")
print(f"Missing values in the dataset:\n{df.isnull().sum()}")
print(f"Data types of the columns:\n{df.dtypes}")

Data needs small cleaning:
1. Drop `id` column (it's just a row index, not a feature).
2. Convert `age` from days to years (age in days / 365).
3. Keep `height` and `weight` as-is (cm and kg). No missing values to drop.

In [ ]:
df = df.drop('id', axis=1)
df['age'] = (df['age'] / 365).round().astype(int)
print(f"After cleaning:\n{df.info()}")

In [ ]:
print(f"Statistical summary of the dataset:\n{df.describe()}")

In [ ]:
X = df.drop('cardio', axis=1)
y = df['cardio']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
Dtree = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)

Dtree.fit(X_train, y_train)

In [ ]:
y_pred = Dtree.predict(X_test)

print(f"Score on Testing Data is {accuracy_score(y_test, y_pred)}")
print(f"Score on training Data is {Dtree.score(X_train, y_train)}")

In [ ]:
for i in range(1, 21):
    trees = DecisionTreeClassifier(criterion='gini', max_depth=i, random_state=42)
    trees.fit(X_train, y_train)
    y_pred = trees.predict(X_test)
    print(f"Max Depth: {i}, Score on Test data: {accuracy_score(y_test, y_pred)}, Score on training data: {accuracy_score(y_train, trees.predict(X_train))}")

3 Depth Given the best result after depth 13 its all over fitting

In [ ]:
from sklearn import tree
plt.figure(figsize=(12,8), dpi= 600)
plt.title("Decision Tree for Cardio Disease Prediction")
tree.plot_tree(Dtree, feature_names=X.columns.tolist(), class_names=['No Cardio', 'Cardio'], filled=True)

plt.show()

In [ ]:
feat_importances = pd.Series(Dtree.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title("Feature Importance in Decision Tree for Cardio Disease Prediction")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n{cm}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")

### Classification Report — Class by Class

**Class 1 (Cardio):** TP=?, FP=?, FN=? | **Class 0 (No):** flip roles → TP=?, FP=?, FN=?

| Metric | Formula | Class 1 | Class 0 |
|---|---|---|---|
| Precision | TP/(TP+FP) | ?/? = **?** | ?/? = **?** |
| Recall | TP/(TP+FN) | ?/? = **?** | ?/? = **?** |
| F1 | 2PR/(P+R) | **?** | **?** |
| Support | actual count | ? | ? |

- **Precision** → when model says this class, how often right?
- **Recall** → of all real cases, how many caught?
- **F1** → balance of both.
- **macro avg** = plain mean · **weighted avg** = weighted by support

**Takeaway:** fill the TP/FP/FN values from the Confusion Matrix cell above.
For medical data, recall on class 1 matters most → fix with SMOTE, `max_depth` tuning, or threshold adjustment.

In [ ]:
group_names = ['True Neg', 'False Pos', 'False Neg', 'True Pos']
group_counts = [f"{v}" for v in cm.flatten()]
labels = np.asarray([f"{n}\n{c}" for n, c in zip(group_names, group_counts)]).reshape(2, 2)

sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
            xticklabels=['Pred: No Cardio', 'Pred: Cardio'],
            yticklabels=['Actual: No Cardio', 'Actual: Cardio'])
plt.show()

In [ ]:
import pickle
with open('Dtree_cardio_model.pkl', 'wb') as f:
    pickle.dump(Dtree, f)